Transformer block from scratch

Build the core components: Multi-Head Attention, FFN, and LayerNorm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# scaled dot product attention
def attention(Q,K,V,mask=None):
  """
    Q, K, V: [batch, heads, seq_len, d_k]
    mask: [1, 1, seq_len, seq_len] boolean - True=attend, False=ignore.
          For causal (autoregressive) masking, use torch.tril(torch.ones(seq, seq)).
    Returns: [batch, heads, seq_len, d_k]
  """
  d_k = Q.size(-1)

  # compute attention scores
  scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(d_k)

  # apply mask (for causal attention)
  if mask is not None:
    scores = scores.masked_fill(mask == 0, float('inf'))


  # softmax -> weights sum to 1
  weights = F.softmax(scores, dim=-1)

  return torch.matmul(weight,V)

# Multi Head Attention

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model = 512, n_heads = 8):
    super().__init__()
    self.n_heads = n_heads
    self.d_k = d_model // n_heads

    # Projections for Q, K, V
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def forward(self, x, mask=None):
    batch, seq_len, _ = x.shape

    # Project and reshape: [batch, seq, d_model] → [batch, heads, seq, d_k]
    Q = self.W_q(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1,2)
    K = self.W_k(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
    V = self.W_v(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)

    # apply attention
    attn_out = attention(Q,K,V,mask)

    attn_out = att_out.transpose(1,2).contingous().view(batch, seq_len, -1)

    return self.W_o(attn_out)
